# Comparing Spline Bases: cr() vs cc()

This notebook demonstrates the difference between natural cubic splines (`cr()`) and cyclic cubic splines (`cc()`) using the UCI Bike Sharing dataset.

**Key concepts:**
- `cr()` (natural cubic spline): Best for smooth trends where boundary behavior matters but isn't periodic
- `cc()` (cyclic cubic spline): Best for periodic/circular predictors where endpoints should "wrap around"

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from formulae import design_matrices

## The Data: Bike Sharing Rentals

We'll use a subset of the UCI Bike Sharing dataset. This dataset records hourly bike rental counts along with weather and seasonal information.

Two predictors make this dataset perfect for comparing spline types:
- **Hour of day (0-23)**: Naturally cyclic — hour 23 is adjacent to hour 0
- **Temperature**: Continuous, non-periodic — warm temperatures don't "wrap" to cold

In [ ]:
# Embedded sample from UCI Bike Sharing Dataset (hourly)
# Source: https://archive.ics.uci.edu/ml/datasets/bike+sharing+dataset
# This is a representative subset for demonstration purposes

np.random.seed(42)

# Simulate realistic bike sharing patterns
n_days = 30
hours = np.tile(np.arange(24), n_days)
days = np.repeat(np.arange(n_days), 24)

# Temperature: varies by hour and day (warmer midday, cooler at night)
temp_base = 15 + 5 * np.sin(2 * np.pi * (days / 30))  # Seasonal trend
temp_daily = 5 * np.sin(2 * np.pi * (hours - 6) / 24)  # Daily cycle
temp = temp_base + temp_daily + np.random.normal(0, 2, len(hours))
temp = np.clip(temp, 0, 35)  # Realistic range in Celsius

# Bike rentals: peaks at commute hours (8am, 6pm), affected by temperature
# This creates a pattern where hour-of-day matters AND temperature matters
hour_effect = (
    80 * np.exp(-0.5 * ((hours - 8) / 2) ** 2) +   # Morning commute
    100 * np.exp(-0.5 * ((hours - 17) / 2) ** 2) +  # Evening commute  
    30 * np.exp(-0.5 * ((hours - 12) / 3) ** 2)     # Midday activity
)
temp_effect = 5 * temp - 0.1 * (temp - 20) ** 2  # Optimal around 20C
noise = np.random.normal(0, 20, len(hours))

rentals = np.maximum(0, hour_effect + temp_effect + noise).astype(int)

bike_data = pd.DataFrame({
    'hour': hours,
    'day': days,
    'temp': temp,
    'rentals': rentals
})

print(f"Dataset: {len(bike_data)} hourly observations over {n_days} days")
bike_data.head(10)

## Visualizing the Patterns

Let's look at how rentals vary by hour and temperature:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hour of day pattern
hourly_avg = bike_data.groupby('hour')['rentals'].mean()
axes[0].bar(hourly_avg.index, hourly_avg.values, alpha=0.7, color='steelblue')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Average Rentals')
axes[0].set_title('Bike Rentals by Hour\n(Notice: Hour 23 should connect to Hour 0)')
axes[0].axvline(x=-0.5, color='red', linestyle='--', alpha=0.5, label='Wrap point')
axes[0].axvline(x=23.5, color='red', linestyle='--', alpha=0.5)

# Temperature pattern
axes[1].scatter(bike_data['temp'], bike_data['rentals'], alpha=0.3, s=10, color='coral')
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Rentals')
axes[1].set_title('Bike Rentals by Temperature\n(No wrap-around needed)')

plt.tight_layout()
plt.show()

## Why Cyclic Splines for Hour?

Hour of day is **circular**: hour 23 is just one hour before hour 0. If we use a regular spline, the fitted curve at hour 23 has no knowledge of hour 0, potentially creating unrealistic discontinuities.

Let's compare the two spline types on the hour predictor:

In [ ]:
# Build design matrices with both spline types for hour
dm_natural = design_matrices("cr(hour, df=8) - 1", bike_data)
dm_cyclic = design_matrices("cc(hour, df=8, lower_bound=0, upper_bound=24) - 1", bike_data)

print("Natural cubic spline (cr) basis shape:", dm_natural.common.design_matrix.shape)
print("Cyclic cubic spline (cc) basis shape:", dm_cyclic.common.design_matrix.shape)

In [ ]:
# Visualize the basis functions
hours_fine = np.linspace(0, 24, 200)
data_fine = pd.DataFrame({'hour': hours_fine})

# Evaluate basis on fine grid
basis_natural = dm_natural.common.evaluate_new_data(data_fine).design_matrix
basis_cyclic = dm_cyclic.common.evaluate_new_data(data_fine).design_matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Natural spline basis
for i in range(basis_natural.shape[1]):
    axes[0].plot(hours_fine, basis_natural[:, i], label=f'Basis {i}')
axes[0].set_xlabel('Hour of Day')
axes[0].set_ylabel('Basis Value')
axes[0].set_title('Natural Cubic Spline Basis (cr)\nNote: No wrap-around at boundaries')
axes[0].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[0].axvline(x=24, color='gray', linestyle=':', alpha=0.5)
axes[0].legend(loc='upper right', fontsize=8)

# Cyclic spline basis
for i in range(basis_cyclic.shape[1]):
    axes[1].plot(hours_fine, basis_cyclic[:, i], label=f'Basis {i}')
axes[1].set_xlabel('Hour of Day')
axes[1].set_ylabel('Basis Value')
axes[1].set_title('Cyclic Cubic Spline Basis (cc)\nNote: Basis wraps around at 0/24')
axes[1].axvline(x=0, color='gray', linestyle=':', alpha=0.5)
axes[1].axvline(x=24, color='gray', linestyle=':', alpha=0.5)
axes[1].legend(loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

### Verifying Cyclic Continuity

The key property of `cc()`: basis values at hour 0 equal basis values at hour 24 (the period boundary).

In [ ]:
# Check that cyclic basis matches at boundaries
boundary_data = pd.DataFrame({'hour': [0.0, 24.0]})

basis_at_boundaries = dm_cyclic.common.evaluate_new_data(boundary_data).design_matrix

print("Cyclic spline basis at hour=0:")
print(basis_at_boundaries[0])
print("\nCyclic spline basis at hour=24:")
print(basis_at_boundaries[1])
print("\nDifference (should be ~0):")
print(basis_at_boundaries[0] - basis_at_boundaries[1])

## Why Natural Splines for Temperature?

Temperature is **not cyclic** — 0°C is not adjacent to 35°C. However, we still want:
- A smooth, flexible fit through the data
- Sensible behavior at the boundaries (no wild extrapolation)

Natural cubic splines (`cr()`) are ideal because they constrain the second derivative to zero at boundaries, making the fit linear beyond the observed range.

In [ ]:
# Build design matrix for temperature
dm_temp = design_matrices("cr(temp, df=6) - 1", bike_data)

print("Natural cubic spline basis for temperature:")
print(f"Shape: {dm_temp.common.design_matrix.shape}")
print(f"\nColumn names: {dm_temp.common.terms['cr(temp, df=6)'].labels}")

In [ ]:
# Visualize temperature basis
temp_fine = np.linspace(bike_data['temp'].min() - 2, bike_data['temp'].max() + 2, 200)
temp_data_fine = pd.DataFrame({'temp': temp_fine})

basis_temp = dm_temp.common.evaluate_new_data(temp_data_fine).design_matrix

fig, ax = plt.subplots(figsize=(10, 5))

for i in range(basis_temp.shape[1]):
    ax.plot(temp_fine, basis_temp[:, i], label=f'Basis {i}', linewidth=2)

# Mark the observed data range
ax.axvline(x=bike_data['temp'].min(), color='gray', linestyle='--', alpha=0.5, label='Data range')
ax.axvline(x=bike_data['temp'].max(), color='gray', linestyle='--', alpha=0.5)

ax.set_xlabel('Temperature (°C)')
ax.set_ylabel('Basis Value')
ax.set_title('Natural Cubic Spline Basis for Temperature (cr)\nNote: Linear behavior outside data range')
ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## Combining Both in a Model

The real power comes from using both spline types together. Here's a formula that models bike rentals as a function of:
- Hour of day (cyclic, because 11pm → midnight → 1am)
- Temperature (natural, smooth but not periodic)

In [ ]:
# Combined model formula
formula = "rentals ~ cc(hour, df=8, lower_bound=0, upper_bound=24) + cr(temp, df=6)"

dm_combined = design_matrices(formula, bike_data)

print(dm_combined.common)
print(f"\nTotal design matrix shape: {dm_combined.common.design_matrix.shape}")
print(f"(1 intercept + 8 hour basis + 6 temp basis = 15 columns)")

In [ ]:
# Show the design matrix for first few observations
dm_combined.common.as_dataframe().head(10)

## Stateful Transforms: Prediction on New Data

Both `cr()` and `cc()` are **stateful transforms**. This means:

1. During fitting, they compute and store knot locations and bounds from the training data
2. During prediction, they reuse these stored parameters on new data

This is critical for proper out-of-sample prediction!

In [ ]:
# Simulate "new" data for prediction
new_data = pd.DataFrame({
    'hour': [6, 12, 18, 23, 0],  # Various hours including boundary
    'temp': [10, 20, 25, 15, 5]   # Various temperatures
})

print("New data for prediction:")
print(new_data)

# Evaluate using stored parameters from training
new_common = dm_combined.common.evaluate_new_data(new_data)

print("\nDesign matrix for new data (using knots from training):")
new_common.as_dataframe()

In [ ]:
# Verify that the stored knots are from the original data
cr_term = dm_combined.common.terms['cr(temp, df=6)'].components[0].call.stateful_transform
cc_term = dm_combined.common.terms['cc(hour, df=8, lower_bound=0, upper_bound=24)'].components[0].call.stateful_transform

print("Stored parameters for cr(temp, df=6):")
print(f"  Knots: {cr_term._knots}")
print(f"  Lower bound: {cr_term._lower_bound:.2f}")
print(f"  Upper bound: {cr_term._upper_bound:.2f}")

print("\nStored parameters for cc(hour, df=8):")
print(f"  Knots: {cc_term._knots}")
print(f"  Lower bound: {cc_term._lower_bound}")
print(f"  Upper bound: {cc_term._upper_bound}")

## Summary

| Transform | Use When | Key Property |
|-----------|----------|-------------|
| `cr()` | Smooth, non-periodic predictors | Linear at boundaries (stable extrapolation) |
| `cc()` | Circular/periodic predictors | Wraps around (endpoint continuity) |

**Common parameters:**
- `df`: Degrees of freedom (number of basis functions)
- `knots`: Explicit interior knot locations (optional)
- `lower_bound`, `upper_bound`: Boundary locations (defaults to data min/max)

**Note:** These transforms provide the basis matrix only. For automatic smoothing penalty estimation (like R's mgcv), use these bases within Bambi or another library that handles penalization.